# Avance 3 — Análisis Final e Insights

**Estudiante:** Lourdes Diamela Alarcon

_Actualizado: 2025-09-17_

## Descripción
Se analizan de forma conjunta los datos de clientes, los agregados de Yelp y la señal de tendencia externa. Se incluyen visualizaciones, correlaciones y una segmentación descriptiva. Las conclusiones orientan recomendaciones para la segmentación de campañas.


In [ ]:

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

ENRICHED = '../data/processed_clientes_enriched_trends.csv'
df = pd.read_csv(ENRICHED)
print('Shape:', df.shape)
df.head(5)


### Correlaciones (incluyendo la variable de tendencia externa)

In [ ]:

num_cols = [c for c in ['edad','frecuencia_visita','promedio_gasto_comida','yelp_rating_ciudad_prom','trend_index_city_mean'] if c in df.columns]
corr = df[num_cols].corr()
corr


In [ ]:

# Heatmap con ejes legibles
fig, ax = plt.subplots(figsize=(6,5))
cax = ax.imshow(corr, interpolation='nearest')
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=30, ha='right'); ax.set_yticklabels(corr.columns)
fig.colorbar(cax); plt.title('Matriz de correlación')
plt.tight_layout(); plt.savefig('../reports/figures/corr_heatmap.png', bbox_inches='tight'); plt.show()


**Lectura:** valores positivos sugieren asociación directa entre variables; negativos, relación inversa. Esta vista guía qué combinaciones revisar en mayor detalle.

### Segmentación por gasto y frecuencia

In [ ]:

df_seg = df.copy()
if 'promedio_gasto_comida' in df_seg.columns:
    df_seg['seg_monetary'] = pd.qcut(df_seg['promedio_gasto_comida'].rank(method='first'), 3, labels=['Bajo','Medio','Alto'])
if 'frecuencia_visita' in df_seg.columns:
    df_seg['seg_frequency'] = pd.qcut(df_seg['frecuencia_visita'].rank(method='first'), 3, labels=['Bajo','Medio','Alto'])

segment_table = df_seg.groupby(['seg_monetary','seg_frequency'], dropna=False).size().unstack(fill_value=0)
segment_table


**Lectura:** la tabla resume el tamaño relativo de cada combinación gasto×frecuencia; sirve para priorizar tácticas por volumen y valor potencial.

### Yelp vs. segmentos (rating promedio por celda)

In [ ]:

if {'seg_monetary','seg_frequency','yelp_rating_ciudad_prom'}.issubset(df_seg.columns):
    rel = df_seg.groupby(['seg_monetary','seg_frequency'])['yelp_rating_ciudad_prom'].mean().unstack()
    rel



### Recomendaciones
- **Alto gasto + Alta frecuencia**: beneficios y experiencias exclusivas; mantener lealtad.
- **Alto gasto + Baja frecuencia**: recordatorios y prueba social (reseñas Yelp); facilitar reservas.
- **Bajo gasto + Alta frecuencia**: programa de puntos y upgrades por recurrencia.
- **Bajo gasto + Baja frecuencia**: ofertas de entrada y referidos para activar.
